# 법률 문서 임베딩 학습 및 평가

한국어 Sentence-BERT 모델을 이용해 판례 검색 기준선을 만들고, 유사 문장 쌍으로 선택적 미세조정을 수행함. 현재 포함된 데이터는 기능 검증용 합성 데이터이므로 최종 실험에서는 출처가 명확한 실제 판례로 교체할 예정임.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sentence_transformers import InputExample, SentenceTransformer, losses

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data import documents_from_cases, load_cases

DATA_PATH = ROOT / 'data' / 'sample_cases.csv'
PAIR_PATH = ROOT / 'data' / 'training_pairs.csv'
MODEL_PATH = ROOT / 'models' / 'legal-sbert'
BASE_MODEL = 'jhgan/ko-sroberta-multitask'
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')

## 1. 데이터 확인 및 전처리

In [ ]:
cases = load_cases(DATA_PATH)
pairs = pd.read_csv(PAIR_PATH)
display(cases[['case_name', 'category', 'issues']].head())
print(f'판례: {len(cases):,}건 / 학습 쌍: {len(pairs):,}개')
cases['category'].value_counts().plot.bar(title='사건 분야 분포', rot=30)
plt.tight_layout()
plt.show()

In [ ]:
documents = documents_from_cases(cases)
print(documents[0][:500])

## 2. 기본 모델 검색 성능

각 판례의 `쟁점`을 질의로 사용하고, 동일 판례가 검색 결과 Top-K에 포함되는지 Recall@K로 평가함

In [ ]:
model = SentenceTransformer(BASE_MODEL)
document_embeddings = model.encode(
    documents, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True
)
query_embeddings = model.encode(
    cases['issues'].tolist(), convert_to_numpy=True, normalize_embeddings=True
)

similarities = query_embeddings @ document_embeddings.T

def recall_at_k(score_matrix, k):
    top_indices = np.argsort(score_matrix, axis=1)[:, ::-1][:, :k]
    return np.mean([idx in top_indices[idx] for idx in range(len(top_indices))])

for k in [1, 3, 5]:
    print(f'Recall@{k}: {recall_at_k(similarities, k):.3f}')

## 3. 법률 유사 문장 쌍 미세조정

`RUN_TRAINING=True`로 변경하면 Multiple Negatives Ranking Loss로 모델을 학습함. 실제 실험에서는 학습·검증·테스트 데이터를 사건 단위로 분리하고 수천 개 이상의 판례 쌍을 사용.

In [ ]:
RUN_TRAINING = False
EPOCHS = 1
BATCH_SIZE = 8

if RUN_TRAINING:
    train_examples = [
        InputExample(texts=[row.anchor, row.positive])
        for row in pairs.itertuples(index=False)
    ]
    train_loader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)
    train_loss = losses.MultipleNegativesRankingLoss(model)
    warmup_steps = max(1, int(len(train_loader) * EPOCHS * 0.1))

    model.fit(
        train_objectives=[(train_loader, train_loss)],
        epochs=EPOCHS,
        warmup_steps=warmup_steps,
        output_path=str(MODEL_PATH),
        show_progress_bar=True,
    )
    print('학습 모델 저장:', MODEL_PATH)
else:
    print('학습을 실행하려면 RUN_TRAINING을 True로 변경하세요.')

## 4. 최종 모델 평가 및 검색 예시

In [ ]:
final_model = SentenceTransformer(str(MODEL_PATH) if MODEL_PATH.exists() else BASE_MODEL)
final_document_embeddings = final_model.encode(
    documents, convert_to_numpy=True, normalize_embeddings=True
)

query = '인터넷에서 산 물건이 고장 났는데 판매자가 환불을 해주지 않는다'
query_vector = final_model.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]
scores = final_document_embeddings @ query_vector
top_indices = np.argsort(scores)[::-1][:3]

result = cases.iloc[top_indices][['case_name', 'category', 'issues']].copy()
result.insert(0, 'similarity', scores[top_indices])
display(result)

## 5. Streamlit용 전체 인덱스 저장

학습을 실행한 경우 아래 셀로 앱에서 사용할 임베딩과 판례 메타데이터를 갱신.

In [ ]:
artifact_dir = ROOT / 'artifacts'
artifact_dir.mkdir(exist_ok=True)
np.save(artifact_dir / 'case_embeddings.npy', final_document_embeddings.astype('float32'))
cases.to_csv(artifact_dir / 'indexed_cases.csv', index=False, encoding='utf-8-sig')
print('인덱스 저장 완료:', artifact_dir)